<a href="https://colab.research.google.com/github/Arjuniyer2005/Econ-5200-PS1/blob/main/Problem_Set_1_Econ_5200_PS1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Econ 5200 PS1 — The Measurement Audit

**Starter notebook.** The Canvas assignment is the brief: read it first. This
notebook gives you the structure, one heading per step in the order the rubric
grades them, and the module skeleton the brief asks for. Every result is yours
to compute.

**Before you start**
1. *File › Save a copy in Drive*, and rename the copy `Econ 5200 PS1`.
2. Every claim gets a text cell. A deliverable that cannot be read without
   running it is not a deliverable.
3. **AI is not allowed in Phases 1–3.** Phase 4 requires it, with the
   transcript in `ai-appendix.md` in your repository.

In [17]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)     # one generator, so every draw is reproducible

---
## Phase 1: Reproduce the Wrong Number (15 pts)

### Step 1.1: Build the transaction data

> ### A tool you have not used yet — drawing skewed data
>
> `rng.lognormal(mean, sigma, size)` draws right-skewed positive values: most
> are modest, a few are large. `mean` and `sigma` describe the logarithm of the
> values, not the values themselves, so check `np.median(...)` of a draw
> before you settle on them. `rng.choice(...)` and `rng.integers(...)` are
> useful for the ids and the years.

At least 20,000 rows over two years: `customer_id`, `year`, `basket_value`.
Plant the B2B orders and the change in what gets logged, and record the true
consumer average for each year **before** you contaminate anything.

In [18]:
# Clean consumer transactions
import numpy as np
import pandas as pd

# Reset seed inside this cell to guarantee identical results on every run
rng = np.random.default_rng(42)

n = 20000

consumer_df = pd.DataFrame({
    "customer_id": rng.integers(1000, 6000, size=n),
    "year": rng.choice([1, 2], size=n),
    "basket_value": rng.lognormal(mean=3.8, sigma=0.6, size=n)
})

consumer_df.head()

,customer_id,year,basket_value
0,1446,1,14.947390
1,4869,2,35.275440
2,4272,1,15.657647
3,3194,2,152.158581
4,3165,2,50.779602


In [19]:
# The truth: the consumer average per year, recorded before contamination

true_consumer_avg = consumer_df.groupby("year")["basket_value"].mean()

print(true_consumer_avg)

year
1    53.953552
2    53.973170
Name: basket_value, dtype: float64


In [20]:
# Contamination 1: B2B orders

# Reset seed to guarantee consistent B2B generation
rng_b2b = np.random.default_rng(43)

n_b2b = 100

b2b_orders = pd.DataFrame({
    "customer_id": rng_b2b.integers(6000, 6100, size=n_b2b),
    "year": rng_b2b.choice([1, 2], size=n_b2b),
    "basket_value": rng_b2b.lognormal(mean=7.5, sigma=0.5, size=n_b2b)
})

transactions = pd.concat([consumer_df, b2b_orders], ignore_index=True)

In [21]:
# Contamination 2: what gets logged changes between year 1 and year 2

# Reset seed to guarantee consistent cancelled orders generation
rng_cancelled = np.random.default_rng(44)

n_cancelled = 500

cancelled_orders = pd.DataFrame({
    "customer_id": rng_cancelled.integers(1000, 6000, size=n_cancelled),
    "year": 2,
    "basket_value": rng_cancelled.lognormal(mean=3.8, sigma=0.6, size=n_cancelled)
})

transactions = pd.concat(
    [transactions, cancelled_orders],
    ignore_index=True
)

### Step 1.2: Reproduce the dashboard

Compute the naive `mean(basket_value)` by year, as the dashboard does, and
quantify its gap from the truth.

In [22]:
# The dashboard's number vs the truth
naive_avg = transactions.groupby("year")["basket_value"].mean()

print("Naive dashboard average:")
print(naive_avg.round(2))

naive_yoy = (
    (naive_avg.loc[2] / naive_avg.loc[1]) - 1
) * 100

print(f"Naive dashboard YoY change: {naive_yoy:.2f}%")

true_yoy = (
    (true_consumer_avg.loc[2] / true_consumer_avg.loc[1]) - 1
) * 100

gap = naive_yoy - true_yoy

print(f"True consumer YoY change: {true_yoy:.2f}%")
print(f"Naive dashboard YoY change: {naive_yoy:.2f}%")
print(f"Gap from truth: {gap:.2f} percentage points")

Naive dashboard average:
year
1    61.29
2    65.65
Name: basket_value, dtype: float64
Naive dashboard YoY change: 7.11%
True consumer YoY change: 0.04%
Naive dashboard YoY change: 7.11%
Gap from truth: 7.08 percentage points


---
## Phase 2: Locate the Cause (40 pts)

### Step 2.1: Separate the two mechanisms (25 pts)

Report each contribution as a number.

In [23]:
# How much of the gap is the skew and the B2B tail?
transactions_b2b = pd.concat(
    [consumer_df, b2b_orders],
    ignore_index=True
)

b2b_avg = transactions_b2b.groupby("year")["basket_value"].mean()

b2b_yoy = (
    (b2b_avg.loc[2] / b2b_avg.loc[1]) - 1
) * 100

b2b_contribution = b2b_yoy - true_yoy

print(f"True consumer YoY change: {true_yoy:.2f}%")
print(f"YoY change with B2B included: {b2b_yoy:.2f}%")
print(f"B2B contribution to the gap: {b2b_contribution:.2f} percentage points")

True consumer YoY change: 0.04%
YoY change with B2B included: 8.02%
B2B contribution to the gap: 7.98 percentage points


In [24]:
# How much is the change in what was logged?
logging_contribution = naive_yoy - b2b_yoy

print(f"YoY change with B2B only: {b2b_yoy:.2f}%")
print(f"Final naive dashboard YoY change: {naive_yoy:.2f}%")
print(f"Logging-change contribution: {logging_contribution:.2f} percentage points")

YoY change with B2B only: 8.02%
Final naive dashboard YoY change: 7.11%
Logging-change contribution: -0.90 percentage points


| Mechanism | Contribution to the gap | How you isolated it |
|---|---|---|
| Skew and B2B tail | 7.98 percentage points | Compared the YoY change of the dataset containing consumer data plus B2B orders (`transactions_b2b`) against the true clean consumer YoY change (`true_yoy`). |
| Change in what was logged | -0.90 percentage points | Subtracted the YoY change of the B2B-only contaminated dataset (`b2b_yoy`) from the final naive dashboard YoY change (`naive_yoy`). |

### Step 2.2: Robust alternatives (15 pts)

`stats.trim_mean(x, 0.1)` drops the top and bottom 10% before averaging.

In [25]:
# Median, trimmed mean, and the mean after your B2B exclusion rule

median_by_year = transactions.groupby("year")["basket_value"].median()

trimmed_mean_by_year = transactions.groupby("year")["basket_value"].apply(
    lambda x: stats.trim_mean(x, 0.1)
)

b2b_cutoff = 1000

excluded_b2b_mean_by_year = (
    transactions.loc[transactions["basket_value"] < b2b_cutoff]
    .groupby("year")["basket_value"]
    .mean()
)

print("Median by year:")
print(median_by_year.round(2))

print("\n10% trimmed mean by year:")
print(trimmed_mean_by_year.round(2))

print("\nMean after B2B exclusion:")
print(excluded_b2b_mean_by_year.round(2))

Median by year:
year
1    44.84
2    45.42
Name: basket_value, dtype: float64

10% trimmed mean by year:
year
1    48.75
2    49.11
Name: basket_value, dtype: float64

Mean after B2B exclusion:
year
1    54.54
2    54.86
Name: basket_value, dtype: float64


| Estimator | Value | The assumption that makes it right | When it would be wrong |
|---|---|---|---|
| Median | Year 1: 44.84; Year 2: 45.42 | Appropriate if the goal is to measure a typical consumer transaction while limiting the influence of extreme orders. | Wrong if the business wants average dollars per transaction, because the median ignores the magnitude of large legitimate purchases. |
| Trimmed mean | Year 1: 48.75; Year 2: 49.11 | Appropriate if the most extreme transactions are unrepresentative of normal consumer behavior. | Wrong if the trimmed observations are legitimate consumer purchases that should count in the metric. |
| Mean after B2B exclusion | Year 1: 54.54; Year 2: 54.86 | Appropriate if average basket value is meant to measure consumer transactions and the B2B rule correctly identifies non-consumer orders. | Wrong if the exclusion rule removes legitimate high-value consumer purchases or fails to identify B2B orders. |

The dashboard should show the arithmetic mean after explicitly excluding B2B orders and applying a consistent logging definition across both years. This preserves the intended meaning of “average basket value” as average dollars per consumer transaction rather than replacing it with a different statistic such as the median. In this simulation, the B2B-excluded mean rises from $54.54 in Year 1 to $54.86 in Year 2. The logging inconsistency must also be corrected so that both years contain comparable transaction types.

---
## Phase 3: Ship the Fix (20 pts)

### Step 3.1: The module

The skeleton below fixes the names the brief asks for. Replace each body,
write the docstrings properly, and keep the type hints. Lab 4's
`OutlierDetector` is a model for the shape: one function per job, the method
chosen by a parameter, and no reliance on a global variable.

In [26]:
import os
os.makedirs("src", exist_ok=True)

In [27]:
%%writefile src/basket_metrics.py
"""Utilities for auditing basket-value data and computing robust summary measures."""

import numpy as np
import pandas as pd
from scipy import stats
from typing import Sequence


def audit_report(df: pd.DataFrame) -> pd.DataFrame:
    """Return missingness, dtype, skew, and IQR-based outlier count for each column."""
    rows = []

    for col in df.columns:
        series = df[col]

        missingness = series.isna().sum()
        dtype = series.dtype

        if pd.api.types.is_numeric_dtype(series):
            clean = series.dropna()
            skew = clean.skew()

            q1 = clean.quantile(0.25)
            q3 = clean.quantile(0.75)
            iqr = q3 - q1

            lower = q1 - 1.5 * iqr
            upper = q3 + 1.5 * iqr

            outlier_count = ((clean < lower) | (clean > upper)).sum()
        else:
            skew = np.nan
            outlier_count = np.nan

        rows.append({
            "column": col,
            "missingness": missingness,
            "dtype": str(dtype),
            "skew": skew,
            "outlier_count": outlier_count
        })

    return pd.DataFrame(rows).set_index("column")


def robust_mean(x: Sequence[float], method: str = "median") -> float:
    """Return a robust centre using median, trimmed mean, or B2B exclusion."""
    x = pd.Series(x).dropna()

    if method == "median":
        return float(x.median())

    if method == "trimmed":
        return float(stats.trim_mean(x, 0.1))

    if method == "rule":
        b2b_cutoff = 1000
        return float(x[x < b2b_cutoff].mean())

    raise ValueError("method must be 'median', 'trimmed', or 'rule'")


if __name__ == "__main__":
    demo = pd.DataFrame({
        "basket_value": [40, 45, 50, 55, 60, 5000]
    })

    print("Audit report:")
    print(audit_report(demo))

    print("\nMedian:")
    print(robust_mean(demo["basket_value"], method="median"))

    print("\nTrimmed mean:")
    print(robust_mean(demo["basket_value"], method="trimmed"))

    print("\nMean after B2B exclusion:")
    print(robust_mean(demo["basket_value"], method="rule"))

Overwriting src/basket_metrics.py


### Step 3.2: Verification checkpoint

Import the module back and test it on data it has not seen. An `assert` stops
with an error when its condition is False, which is the point: a failing
assertion tells you something.

In [28]:
import importlib
import sys

sys.path.insert(0, "src")

import basket_metrics
importlib.reload(basket_metrics)


# Test data the module has not seen before

symmetric_data = pd.Series([10, 20, 30, 40, 50])

outlier_data = pd.Series([20, 25, 30, 35, 40, 5000])

test_df = pd.DataFrame({
    "basket_value": [20, 25, 30, 35, 40, 5000],
    "customer_id": [1, 2, 3, 4, 5, 6]
})


# 1. Median of symmetric data should equal its arithmetic mean
assert abs(
    basket_metrics.robust_mean(symmetric_data, method="median")
    - symmetric_data.mean()
) < 1e-9


# 2. Rule-based exclusion should remove the extreme B2B-style order
assert basket_metrics.robust_mean(
    outlier_data, method="rule"
) < outlier_data.mean()


# 3. audit_report should return one row for every input column
report = basket_metrics.audit_report(test_df)

assert len(report) == len(test_df.columns)


print("All verification checks passed.")

All verification checks passed.


---
## Phase 4: AI Expansion (25 pts)

AI is now allowed and required. The transcript goes in `ai-appendix.md` in your
repository, linked from here: the prompt as sent, the raw reply including what
you rejected, and one line on every change you made.

**Link to `ai-appendix.md`:** https://github.com/Arjuniyer2005/Econ-5200-PS1/blob/main/ai-appendix.md

### Task 4.1: The adversarial review (15 pts)

Run the prompt from the brief, then run the strongest objection down.

**The strongest objection, and the evidence that settles it:**
The strongest objection is that using a 10% trimmed mean after explicitly removing B2B transactions may discard legitimate consumer purchases and therefore change the meaning of “average basket value.” If B2B orders are the source of the extreme tail, then the cleaned arithmetic mean may be the more defensible metric.

To settle this, I would recalculate the metric using a consistent population of completed consumer transactions only, then compare the ordinary mean with the 10% trimmed mean. I would also inspect the observations removed by trimming to determine whether they are genuine consumer transactions.

If the cleaned mean and trimmed mean give similar year-over-year results, that would support my Phase 2 recommendation to use the cleaned arithmetic mean. It preserves the business definition of average basket value while directly correcting the known measurement problems instead of removing valid transactions solely because they are extreme.

In [29]:
# The analysis that settles it

# Keep only completed consumer transactions:
# 1. Exclude B2B orders (customer_id >= 6000)
# 2. Exclude the cancelled orders (which are the last 500 records appended to transactions)
completed_consumer = transactions.iloc[:20000]

cleaned_mean = completed_consumer.groupby("year")["basket_value"].mean()

cleaned_trimmed_mean = completed_consumer.groupby("year")["basket_value"].apply(
    lambda x: stats.trim_mean(x, 0.1)
)

cleaned_mean_yoy = (
    (cleaned_mean.loc[2] / cleaned_mean.loc[1]) - 1
) * 100

cleaned_trimmed_yoy = (
    (cleaned_trimmed_mean.loc[2] / cleaned_trimmed_mean.loc[1]) - 1
) * 100

print("Cleaned arithmetic mean:")
print(cleaned_mean.round(2))

print("\nCleaned 10% trimmed mean:")
print(cleaned_trimmed_mean.round(2))

print(f"\nCleaned arithmetic mean YoY: {cleaned_mean_yoy:.2f}%")
print(f"Cleaned trimmed mean YoY: {cleaned_trimmed_yoy:.2f}%")
print(
    f"Difference between the two YoY estimates: "
    f"{abs(cleaned_mean_yoy - cleaned_trimmed_yoy):.2f} percentage points"
)

Cleaned arithmetic mean:
year
1    53.95
2    53.97
Name: basket_value, dtype: float64

Cleaned 10% trimmed mean:
year
1    48.51
2    48.71
Name: basket_value, dtype: float64

Cleaned arithmetic mean YoY: 0.04%
Cleaned trimmed mean YoY: 0.40%
Difference between the two YoY estimates: 0.36 percentage points


**What you found, including if it changes your Phase 2 recommendation:**

The additional analysis supports, rather than changes, my Phase 2 recommendation. After removing the known B2B and logging issues, the 10% trimmed mean still produces a YoY estimate about 0.35 percentage points different from the arithmetic mean. Because the remaining observations are intended to represent valid consumer transactions, trimming would remove genuine high-value baskets and change the business definition of average basket value. I therefore retain the cleaned arithmetic mean as the recommended dashboard metric.

### Task 4.2: The board slide (10 pts)

150 words at most: the headline number, the correction, the confidence, and the
one thing you would still want before betting on it.

Average Basket Value Was Essentially Flat, Not Up 7%+

**Headline:** After removing B2B orders and applying a consistent completed-consumer definition across both years, average basket value increased by only 0.04% year over year, versus the dashboard’s reported 7.11% increase.

**Correction:** The error was driven by two separate issues: B2B transactions inflated the reported growth by about 7.98 percentage points, while the Year 2 logging change reduced it by about 0.90 points. The dashboard should use the cleaned arithmetic mean so valid high-value consumer purchases remain included.

**High Confidence: **The corrected result is based on a known consumer population and directly reconciles the two identified sources of distortion.

**Before betting on it:** Validate the B2B exclusion rule against a true customer-type identifier rather than relying on transaction size alone.

---
### Before you submit
- [ ] Truth recorded before contamination, and both mechanisms reported as numbers
- [ ] `src/basket_metrics.py` pushed with the notebook, and at least three assertions passing
- [ ] `ai-appendix.md` in the repository and linked above
- [ ] *Runtime › Restart and run all* runs without errors